In [1]:
from dotenv import load_dotenv

load_dotenv()

import os
print(os.environ['OPENAI_API_KEY'][:20])

sk-proj-MBKtuEXlVzac


In [3]:
import datetime
from langchain.chat_models import init_chat_model

from langchain_core.messages import HumanMessage, SystemMessage
llm = init_chat_model("gpt-5-mini", model_provider="openai", reasoning_effort="minimal")

In [4]:

query= "Wanted to see list of raised events along with its details in last 24 hours."

prompt="""
The database schema is as follows:

Table severity(severity_id INTEGER PRIMARY KEY, severity_name TEXT, severity_desc TEXT)
Table event(event_id INTEGER PRIMARY KEY, event_name TEXT, event_type TEXT, event_desc TEXT)
Table eventslog(event_id INTEGER, severity_id INTEGER, timestamp DATETIME, status TEXT)

User question: "Wanted to see list of raised events along with its details in last 24 hours."

Generate the SQL query that retrieves event details raised in the last 24 hours, joining relevant tables as needed.

Output: only SQL query compatible for sqlite3
"""

#myprompt=f"prompt" + query

llm_resp = llm.invoke(prompt)
#llm_resp = llm.invoke([SystemMessage(content="You are an expert research planner."),
#        HumanMessage(query=myprompt)]
#)
print (f" LLM response = {llm_resp.content}")

 LLM response = SELECT e.event_id,
       e.event_name,
       e.event_type,
       e.event_desc,
       s.severity_id,
       s.severity_name,
       s.severity_desc,
       l.timestamp,
       l.status
FROM eventslog l
JOIN event e ON l.event_id = e.event_id
LEFT JOIN severity s ON l.severity_id = s.severity_id
WHERE l.timestamp >= datetime('now', '-1 day')
ORDER BY l.timestamp DESC;


In [20]:
llm_resp.content

"SELECT e.event_id,\n       e.event_name,\n       e.event_type,\n       e.event_desc,\n       s.severity_id,\n       s.severity_name,\n       s.severity_desc,\n       l.timestamp,\n       l.status\nFROM eventslog l\nJOIN event e ON l.event_id = e.event_id\nLEFT JOIN severity s ON l.severity_id = s.severity_id\nWHERE l.timestamp >= datetime('now', '-1 day')\nORDER BY l.timestamp DESC;"

In [5]:
import sqlite3
import datetime

def userquery_execute(db_name: str, userquery: str):
    conn = sqlite3.connect(db_name)
    c = conn.cursor()
    c.execute(userquery)
    query_response = c.fetchall()
    conn.close()
    return query_response

userquery_results = userquery_execute("gridevents.db", llm_resp.content)
print(userquery_results)
for row in userquery_results:
        print(row)


[(1, 'Disk Space Low', 'System', 'Disk space is running low on the server', 2, 'Medium', 'Medium severity issue', '2025-11-03T17:27:50.762778', 'Open'), (2, 'Login Failed', 'Security', 'Failed login attempt detected', 3, 'High', 'High severity issue', '2025-11-03T17:27:50.762778', 'Closed'), (3, 'Service Restarted', 'Maintenance', 'A service was restarted successfully', 1, 'Low', 'Low severity issue', '2025-11-03T17:27:50.762778', 'Open')]
(1, 'Disk Space Low', 'System', 'Disk space is running low on the server', 2, 'Medium', 'Medium severity issue', '2025-11-03T17:27:50.762778', 'Open')
(2, 'Login Failed', 'Security', 'Failed login attempt detected', 3, 'High', 'High severity issue', '2025-11-03T17:27:50.762778', 'Closed')
(3, 'Service Restarted', 'Maintenance', 'A service was restarted successfully', 1, 'Low', 'Low severity issue', '2025-11-03T17:27:50.762778', 'Open')


In [14]:

p1= "Wanted to see list of raised events along with its details in last 24 hours."

p2="""
The database schema is as follows:

Table severity(severity_id INTEGER PRIMARY KEY, severity_name TEXT, severity_desc TEXT)
Table event(event_id INTEGER PRIMARY KEY, event_name TEXT, event_type TEXT, event_desc TEXT)
Table eventslog(event_id INTEGER, severity_id INTEGER, timestamp DATETIME, status TEXT)

User question: {p1}

Generate the SQL query that retrieves event details raised in the last 24 hours, joining relevant tables as needed.

Output: only SQL query compatible for sqlite3
""" 

test_prompt = p2.format(p1=p1)
print (test_prompt)




The database schema is as follows:

Table severity(severity_id INTEGER PRIMARY KEY, severity_name TEXT, severity_desc TEXT)
Table event(event_id INTEGER PRIMARY KEY, event_name TEXT, event_type TEXT, event_desc TEXT)
Table eventslog(event_id INTEGER, severity_id INTEGER, timestamp DATETIME, status TEXT)

User question: Wanted to see list of raised events along with its details in last 24 hours.

Generate the SQL query that retrieves event details raised in the last 24 hours, joining relevant tables as needed.

Output: only SQL query compatible for sqlite3



In [22]:
print (userquery_results)
combined_text = ""
for row in userquery_results:
        combined_text += str(row) + "\n" 
        #combined_text = "\n".join(str(row))
print(combined_text)
#combined_text = "\n".join(row for row in userquery_results)        

[(1, 'Disk Space Low', 'System', 'Disk space is running low on the server', 2, 'Medium', 'Medium severity issue', '2025-11-03T17:27:50.762778', 'Open'), (2, 'Login Failed', 'Security', 'Failed login attempt detected', 3, 'High', 'High severity issue', '2025-11-03T17:27:50.762778', 'Closed'), (3, 'Service Restarted', 'Maintenance', 'A service was restarted successfully', 1, 'Low', 'Low severity issue', '2025-11-03T17:27:50.762778', 'Open')]
(1, 'Disk Space Low', 'System', 'Disk space is running low on the server', 2, 'Medium', 'Medium severity issue', '2025-11-03T17:27:50.762778', 'Open')
(2, 'Login Failed', 'Security', 'Failed login attempt detected', 3, 'High', 'High severity issue', '2025-11-03T17:27:50.762778', 'Closed')
(3, 'Service Restarted', 'Maintenance', 'A service was restarted successfully', 1, 'Low', 'Low severity issue', '2025-11-03T17:27:50.762778', 'Open')

